# 🚀 FRIDAY DesktopIntent-0.5B Ultimate (Kaggle High-Capacity Training)

This notebook trains the **Ultimate 0.5B Neural Router** for FRIDAY AI on Kaggle Tesla T4 GPU.

### Key Architecture Highlights:
- **Base Model**: `Qwen/Qwen2.5-0.5B-Instruct`
- **Training Corpus**: ~18,000 samples (Full `synthetic_full` + 3x oversampled `v4_perfect` & `calibration`)
- **LoRA Rank**: `r=64`, `alpha=128` on all linear attention + MLP layers (~15M parameters trained)
- **Context Length**: `512` tokens (zero truncation on complex tools or dynamic schemas)
- **Export Target**: `friday-0.5b-q8.gguf` (525 MB) with sub-250ms real-time inference.

## 1. Install Dependencies & Clone llama.cpp

In [ ]:
!pip install -q -U transformers peft datasets accelerate trl huggingface_hub gguf sentencepiece protobuf torchao
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git

## 2. Verify GPU Hardware Acceleration

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    raise SystemError("GPU accelerator not detected! Enable GPU in Settings -> Accelerator!")

## 3. Load & Blend Master Dataset from Hugging Face

In [ ]:
from datasets import load_dataset, concatenate_datasets

HF_DATASET_ID = "ThisisDarkstar/friday-desktop-intent"
print(f"[*] Streaming dataset configurations from: {HF_DATASET_ID}...")

synthetic = load_dataset(HF_DATASET_ID, "synthetic_full")
calibration = load_dataset(HF_DATASET_ID, "calibration")
v4_perfect = load_dataset(HF_DATASET_ID, "v4_perfect")

print(f"    synthetic_full: {len(synthetic['train']):,} train / {len(synthetic['validation']):,} val")
print(f"    calibration:    {len(calibration['train']):,} train / {len(calibration['validation']):,} val")
print(f"    v4_perfect:     {len(v4_perfect['train']):,} train / {len(v4_perfect['validation']):,} val")

# Blend with 3x oversampling on high-priority calibration & v4 gating to preserve 100% safety precision
train_ds = concatenate_datasets([
    synthetic["train"],
    calibration["train"],
    calibration["train"],
    calibration["train"],
    v4_perfect["train"],
    v4_perfect["train"],
    v4_perfect["train"],
]).shuffle(seed=42)

val_ds = concatenate_datasets([
    synthetic["validation"],
    calibration["validation"],
    v4_perfect["validation"],
]).shuffle(seed=42)

print(f"[+] Master Blended Training Samples:   {len(train_ds):,}")
print(f"[+] Master Blended Validation Samples: {len(val_ds):,}")

## 4. Initialize Qwen2.5-0.5B-Instruct & High-Rank LoRA (r=64)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

BASE_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "./results_0.5b"
MERGED_DIR = "./friday-0.5b-merged"

print(f"[*] Loading Tokenizer: {BASE_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"[*] Loading Base Model: {BASE_MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map={"": 0},  # Pin directly to GPU 0 to prevent PCIe dual-GPU sharding bottleneck
    trust_remote_code=True,
)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False

# High-capacity LoRA (Rank 64, Alpha 128) covering all attention and MLP projections
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 5. Tokenization & Prompt Loss Masking (512 Context Window)

In [ ]:
def format_chatml(batch, max_length=512):
    input_ids_list = []
    labels_list = []

    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        prompt_msgs = messages[:-1]
        prompt_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)

        full_tokens = tokenizer(text, truncation=True, max_length=max_length)
        prompt_tokens = tokenizer(prompt_text, truncation=True, max_length=max_length)

        p_len = len(prompt_tokens["input_ids"])
        labels = [-100] * p_len + full_tokens["input_ids"][p_len:]

        if len(labels) < len(full_tokens["input_ids"]):
            labels.extend([-100] * (len(full_tokens["input_ids"]) - len(labels)))
        elif len(labels) > len(full_tokens["input_ids"]):
            labels = labels[: len(full_tokens["input_ids"])]

        input_ids_list.append(full_tokens["input_ids"])
        labels_list.append(labels)

    return {"input_ids": input_ids_list, "labels": labels_list}

print("[*] Tokenizing datasets...")
tokenized_train = train_ds.map(format_chatml, batched=True, remove_columns=train_ds.column_names)
tokenized_val = val_ds.map(format_chatml, batched=True, remove_columns=val_ds.column_names)
print(f"[+] Tokenized {len(tokenized_train):,} training examples successfully!")

## 6. Execute Fine-Tuning (3 Epochs, Cosine Annealing)

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,       # Effective batch size = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    num_train_epochs=3,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    optim="adamw_torch",
    report_to="none",
)

collator = DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=collator,
)

print(f"[*] Starting DesktopIntent-0.5B Training on {len(tokenized_train):,} samples...")
trainer.train()

## 7. Merge LoRA Adapter & Save Standalone Model

In [ ]:
import os
print(f"[*] Merging LoRA adapter into base model and saving to {MERGED_DIR}...")
model.config.use_cache = True
merged_model = model.merge_and_unload()
os.makedirs(MERGED_DIR, exist_ok=True)
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"[+] Standalone merged model successfully saved to {MERGED_DIR}!")

## 8. Export to Q8_0 GGUF (525 MB)

In [ ]:
print("[*] Exporting to Q8_0 GGUF (~525 MB - ultra-fast inference)...")
!python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outtype q8_0 --outfile ./friday-0.5b-q8.gguf

print("\n[+] Generated GGUF Artifacts:")
!ls -lh *.gguf